# 138. Copy List with Random Pointer

## Topic Alignment
- **Role Relevance**: Deep copying complex data structures with multiple pointer types is crucial for ML model cloning, graph duplication, and state management in distributed systems.
- **Scenario**: Used when cloning computational graphs in deep learning frameworks, duplicating DAGs for parallel processing, or implementing snapshot isolation in databases.

## Metadata Summary
- Source: [LeetCode - Copy List with Random Pointer](https://leetcode.com/problems/copy-list-with-random-pointer/)
- Tags: `Linked List`, `Hash Table`, `Two Pointers`
- Difficulty: Medium
- Recommended Priority: High

## Problem Statement
A linked list of length `n` is given such that each node contains an additional random pointer, which could point to any node in the list, or `null`. Construct a deep copy of the list. The deep copy should consist of exactly `n` brand new nodes, where each new node has its value set to the value of its corresponding original node. Both the `next` and `random` pointer of the new nodes should point to new nodes in the copied list such that the pointers in the original list and copied list represent the same list state. None of the pointers in the new list should point to nodes in the original list.

Input: `head` - the head of a linked list with nodes containing `val`, `next`, and `random` pointers.
Output: The head of the deep copied linked list.
Constraints: `0 <= n <= 1000`, `-10^4 <= Node.val <= 10^4`, `Node.random` is `null` or points to some node in the list.

## Progressive Hints
- Hint 1: The challenge is mapping old nodes to new nodes. Consider using a hash map where keys are original nodes and values are their copies.
- Hint 2: Make two passes: first create all new nodes with correct values, then set up next and random pointers using the mapping.
- Hint 3: Alternative O(1) space approach: interweave new nodes between original nodes (A->A'->B->B'), then extract the copied list.
- Hint 4: For the interweave approach, first insert copies, then set random pointers using `node.next.random = node.random.next`, finally separate the lists.

## Solution Overview
Two approaches exist: hash map and interweaving. The hash map approach creates a mapping from old nodes to new nodes in the first pass, then connects next and random pointers in the second pass. The interweaving approach inserts each copy immediately after its original, sets random pointers using the interweaved structure, then separates the lists. Hash map is cleaner but uses O(n) space; interweaving achieves O(1) space.

## Detailed Explanation
**Hash Map Approach:**
1. First pass: Traverse the original list and create a new node for each original node. Store the mapping `old_node -> new_node` in a hash map.
2. Second pass: For each original node, set `new_node.next = map[old_node.next]` and `new_node.random = map[old_node.random]` (handle None cases).
3. Return `map[head]` as the new head.

**Interweaving Approach (O(1) space):**
1. First pass: For each node A, create A' and insert it between A and A.next, resulting in A->A'->B->B'->C->C'.
2. Second pass: Set random pointers for copies. For each original node A, if A.random exists, set `A.next.random = A.random.next` (the copy of A.random).
3. Third pass: Separate the lists by restoring original list's next pointers and extracting the copied list.
4. Return the head of the copied list.

The interweaving approach cleverly uses the list structure itself as a hash map, where each original node's next pointer temporarily points to its copy.

## Complexity Trade-off Table
| Approach | Time Complexity | Space Complexity | Notes |
| --- | --- | --- | --- |
| Hash Map | O(n) | O(n) | Cleaner code; requires extra hash map storage. |
| Interweaving | O(n) | O(1) | More complex pointer manipulation; optimal space. |

## Reference Implementation

In [ ]:
from typing import Optional


class Node:
    """Definition for a Node with random pointer."""
    def __init__(self, x: int, next: 'Node' = None, random: 'Node' = None):
        self.val = int(x)
        self.next = next
        self.random = random


def copyRandomList(head: Optional[Node]) -> Optional[Node]:
    """Copy list with random pointer using hash map approach."""
    if not head:
        return None
    
    # First pass: Create all nodes and store mapping
    old_to_new = {}
    current = head
    while current:
        old_to_new[current] = Node(current.val)
        current = current.next
    
    # Second pass: Connect next and random pointers
    current = head
    while current:
        if current.next:
            old_to_new[current].next = old_to_new[current.next]
        if current.random:
            old_to_new[current].random = old_to_new[current.random]
        current = current.next
    
    return old_to_new[head]


def copyRandomListInterweave(head: Optional[Node]) -> Optional[Node]:
    """Copy list with random pointer using interweaving approach (O(1) space)."""
    if not head:
        return None
    
    # First pass: Create interweaved list (A->A'->B->B'->C->C')
    current = head
    while current:
        new_node = Node(current.val, current.next)
        current.next = new_node
        current = new_node.next
    
    # Second pass: Set random pointers for copies
    current = head
    while current:
        if current.random:
            current.next.random = current.random.next
        current = current.next.next
    
    # Third pass: Separate the two lists
    current = head
    new_head = head.next
    while current:
        new_node = current.next
        current.next = new_node.next
        if new_node.next:
            new_node.next = new_node.next.next
        current = current.next
    
    return new_head

## Validation

In [ ]:
def create_list_with_random(values_and_randoms):
    """Helper to create a list with random pointers.
    values_and_randoms: list of tuples (value, random_index or None)
    """
    if not values_and_randoms:
        return None
    
    # Create all nodes first
    nodes = [Node(val) for val, _ in values_and_randoms]
    
    # Connect next pointers
    for i in range(len(nodes) - 1):
        nodes[i].next = nodes[i + 1]
    
    # Connect random pointers
    for i, (_, random_idx) in enumerate(values_and_randoms):
        if random_idx is not None:
            nodes[i].random = nodes[random_idx]
    
    return nodes[0]


def verify_copy(original, copied):
    """Verify that copied list is a deep copy with correct structure."""
    # Check that nodes are different objects
    orig_nodes = set()
    current = original
    while current:
        orig_nodes.add(id(current))
        current = current.next
    
    # Verify copied nodes are new and values/pointers match
    orig_curr = original
    copy_curr = copied
    node_map = {}
    
    # First pass: verify structure and build mapping
    while orig_curr and copy_curr:
        assert id(copy_curr) not in orig_nodes, "Copy shares node with original"
        assert orig_curr.val == copy_curr.val, "Values don't match"
        node_map[id(orig_curr)] = copy_curr
        orig_curr = orig_curr.next
        copy_curr = copy_curr.next
    
    assert orig_curr is None and copy_curr is None, "Length mismatch"
    
    # Second pass: verify random pointers
    orig_curr = original
    copy_curr = copied
    while orig_curr:
        if orig_curr.random:
            expected_random = node_map[id(orig_curr.random)]
            assert copy_curr.random == expected_random, "Random pointer mismatch"
        else:
            assert copy_curr.random is None, "Random should be None"
        orig_curr = orig_curr.next
        copy_curr = copy_curr.next
    
    return True


# Test cases
test_cases = [
    [(7, None), (13, 0), (11, 4), (10, 2), (1, 0)],  # Example 1
    [(1, 1), (2, 1)],  # Example 2
    [(3, None), (3, 0), (3, None)],  # Example 3
    [],  # Empty list
]

for values_and_randoms in test_cases:
    original = create_list_with_random(values_and_randoms)
    
    # Test hash map approach
    copied = copyRandomList(original)
    if values_and_randoms:
        assert verify_copy(original, copied), "Hash map approach failed"
    else:
        assert copied is None, "Empty list should return None"
    
    # Test interweave approach
    original = create_list_with_random(values_and_randoms)  # Recreate original
    copied = copyRandomListInterweave(original)
    if values_and_randoms:
        assert verify_copy(original, copied), "Interweave approach failed"
    else:
        assert copied is None, "Empty list should return None"

print('All tests passed for LC 138.')

## Complexity Analysis
- Time Complexity: O(n) for both approaches where n is the number of nodes. Hash map makes two passes; interweaving makes three passes.
- Space Complexity: O(n) for hash map (storing node mappings), O(1) for interweaving (only using temporary pointers).
- Primary Bottleneck: For very large lists (n = 1000), hash map approach doubles memory usage temporarily, while interweaving maintains constant extra space.

## Edge Cases & Pitfalls
- Empty list (head is None) should return None.
- Random pointer can be None, must handle this case when copying.
- Random pointer can point to the node itself (self-loop).
- In interweaving approach, must carefully restore original list structure in the separation phase.
- When setting random pointers in interweaving, must check if random exists before accessing random.next.

## Follow-up Variants
- Clone a graph where each node can have multiple neighbors (LC 133: Clone Graph).
- Deep copy a binary tree with parent pointers requiring similar mapping techniques.
- Serialize and deserialize the list with random pointers for network transmission.
- Implement copy-on-write semantics for the data structure to delay actual copying.

## Takeaways
- Hash maps are natural for node mapping in graph/list copying problems.
- Interweaving original and copied nodes can eliminate the need for external mapping.
- When copying structures with multiple pointer types, handle each pointer type separately.
- Always verify deep copy independence: no node in the copy should be the same object as in the original.

## Similar Problems
| Problem ID | Problem Title | Technique |
| --- | --- | --- |
| 133 | Clone Graph | Hash map for node duplication |
| 1485 | Clone Binary Tree With Random Pointer | Tree copying with random pointers |
| 1490 | Clone N-ary Tree | N-ary tree deep copy |